# Notebook 05 — Freeze the Audited Pakistani Vehicle Split

Use this **after** running Notebook 04 and reviewing its reported counts.

**Reported from the local cleaned candidate (not yet validated by this notebook):**
- `train: 2,610`, `val: 463`, `test: 751`, `excluded_test_overlap: 23`
- 48 labels in each of train, val and test
- Minimum per class: train 11, val 2, test 3

**Goal:** Independently verify the candidate manifest, reviewer judgments, publisher test membership, and confirmed-pair boundaries. Then export immutable, versionable `final_manifest.csv`, `class_to_idx.json`, and `split_metadata.json`.

The final manifest contains **all 3,847 rows**, with excluded original-training observations marked `excluded_test_overlap`. Future PyTorch loaders must select only rows whose `split` equals `train`, `val`, or `test`.

We do not rename, relocate, delete, or modify image files. A small dHash screen and visual review cannot guarantee that all related road-video frames were found; the final split is **overlap-screened, not proven leakage-free**.

## 1. Read files and verify reproducible hashes

A checksum, such as SHA-256, is a fingerprint of a file. If even one byte changes, the checksum changes, revealing that the data used for training could have changed.

We will verify that the reconciliation notebook's CSV matches its recorded checksum before freezing anything.

In [ ]:
from pathlib import Path
from hashlib import sha256
from collections import Counter
import json
import shutil
import pandas as pd
import yaml

cwd = Path.cwd().resolve()
REPO = next((p for p in (cwd, *cwd.parents) if (p / "configs" / "experiments.yaml").exists()), None)
if REPO is None:
    raise RuntimeError("Open this notebook from within the project repository.")

cfg = yaml.safe_load((REPO / "configs" / "experiments.yaml").read_text(encoding="utf-8"))
split_dir = REPO / cfg["datasets"]["pakistan"]["split_manifest_dir"]

original_path = split_dir / "candidate_manifest.csv"
candidate_path = split_dir / "candidate_clean_manifest.csv"
candidate_meta_path = split_dir / "candidate_clean_metadata.json"
original_meta_path = split_dir / "candidate_split_metadata.json"
review_path = split_dir / "candidate_pair_review.csv"
class_map_path = split_dir / "candidate_class_to_idx.json"

for path in [original_path, candidate_path, candidate_meta_path, original_meta_path, review_path, class_map_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required input: {path}. Run Notebooks 02–04 first.")

sha = lambda p: sha256(p.read_bytes()).hexdigest()
candidate_meta = json.loads(candidate_meta_path.read_text(encoding="utf-8"))
original_meta = json.loads(original_meta_path.read_text(encoding="utf-8"))
assert candidate_meta["status"] == "CLEAN_CANDIDATE_AWAITING_APPROVAL"
assert original_meta["status"] == "CANDIDATE_NOT_APPROVED"
assert sha(original_path) == original_meta["manifest_sha256"]
assert candidate_meta["source_manifest_sha256"] == sha(original_path)
assert candidate_meta["clean_manifest_sha256"] == sha(candidate_path)

orig = pd.read_csv(original_path).fillna("")
clean = pd.read_csv(candidate_path).fillna("")
review = pd.read_csv(review_path).fillna("")
class_map = json.loads(class_map_path.read_text(encoding="utf-8"))

print("Candidate checksum validated:", sha(candidate_path))
print("Rows:", len(clean))
print("Reviewer verdict counts:")
display(review["verdict"].value_counts(dropna=False).rename("pairs").to_frame())

## 2. Apply final integrity assertions

Checks:
- exact same 3,847 relative image paths in original and cleaned manifests;
- 48 classes with stable integer indices 0–47;
- original publisher **test set exactly unchanged**, including filenames and labels;
- every training-origin image assigned only to train/val/excluded;
- no reviewed, confirmed image pair remains on opposite active train/val/test boundaries;
- every excluded training image is connected through confirmed overlap edges to publisher test (computed conservatively from reviewer edges);
- each split retains all 48 classes;
- saved metadata matches observed counts;
- original image files still exist on disk.

If any check fails, the notebook stops before exporting final manifests.

In [ ]:
expected_labels = sorted(class_map)
assert len(class_map) == 48
assert [class_map[k] for k in expected_labels] == list(range(48)), "Class indices must be stable 0..47."

key = "relative_path"
assert set(clean.columns) >= {key,"split","publisher_split","class_name","class_idx"}
assert len(orig) == len(clean) == 3847
assert orig[key].is_unique and clean[key].is_unique
assert set(orig[key]) == set(clean[key])
assert set(clean["class_name"]) == set(expected_labels)
assert set(clean["class_idx"]) == set(range(48))
assert (clean.apply(lambda row: class_map[row["class_name"]] == int(row["class_idx"]), axis=1)).all()
assert set(clean["split"]) == {"train","val","test","excluded_test_overlap"}
assert set(clean["publisher_split"]) == {"train","test"}

by_path = clean.set_index(key)
for row in orig.itertuples(index=False):
    assert by_path.at[row.relative_path,"publisher_split"] == row.publisher_split
    assert by_path.at[row.relative_path,"class_name"] == row.class_name
test_orig = set(orig.loc[orig["publisher_split"].eq("test"), key])
test_final = set(clean.loc[clean["split"].eq("test"), key])
assert test_orig == test_final, "Original publisher test allocation changed."
assert not clean.loc[clean["publisher_split"].eq("train"),"split"].eq("test").any()
assert clean.loc[clean["publisher_split"].eq("test"),"split"].eq("test").all()
assert not clean.loc[clean["split"].eq("excluded_test_overlap"),"publisher_split"].ne("train").any()

allowed = {"same_photo_or_adjacent_frame","likely_same_vehicle","different"}
assert not review.empty and review["review_id"].is_unique
assert set(review["verdict"]).issubset(allowed), "Incomplete or uncertain manual reviews."
assert len(review) == candidate_meta["reviewed_pair_count"]
assert len(review) == original_meta["publisher_train_vs_test_candidate_pairs"] + original_meta["candidate_train_vs_val_candidate_pairs"]

parent = {p:p for p in clean[key]}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a,b):
    a,b=find(a),find(b)
    if a!=b:
        parent[max(a,b)] = min(a,b)

confirmed = review[review["verdict"].isin(["same_photo_or_adjacent_frame","likely_same_vehicle"])]
assert len(confirmed) == candidate_meta["confirmed_pairs"]
for row in confirmed.itertuples(index=False):
    assert row.left_path in parent and row.right_path in parent
    assert by_path.at[row.left_path,"class_name"] == by_path.at[row.right_path,"class_name"]
    union(row.left_path,row.right_path)

linked_to_test = {find(p) for p in test_orig}
expected_excluded = {
    p for p in parent if find(p) in linked_to_test and
    by_path.at[p,"publisher_split"] == "train"
}
actual_excluded = set(clean.loc[clean["split"].eq("excluded_test_overlap"),key])
assert expected_excluded == actual_excluded, "Excluded paths mismatch with reviewed test-linked groups."
assert len(actual_excluded) == candidate_meta["excluded_distinct_publisher_training_images"]

groups = {}
for p in parent:
    groups.setdefault(find(p), set()).add(by_path.at[p,"split"])
for group_splits in groups.values():
    # Test-linked groups may contain both test and excluded images.
    if "test" in group_splits:
        assert group_splits <= {"test", "excluded_test_overlap"}, group_splits
    else:
        assert not ({"train","val"} <= group_splits), group_splits

for rel in clean[key]:
    if not (REPO / rel).is_file():
        raise FileNotFoundError(f"Image missing from local dataset: {rel}")

sizes = clean["split"].value_counts().to_dict()
assert sizes == candidate_meta["counts"], "Cleaned counts differ from reconciliation metadata."
assert sizes["test"] == 751
coverage = clean.loc[clean["split"].isin(["train","val","test"])].groupby("split")["class_name"].nunique().to_dict()
assert coverage == {"train":48,"val":48,"test":48}
mins = pd.crosstab(clean["class_name"],clean["split"])[["train","val","test"]].min().to_dict()
assert all(v>=1 for v in mins.values())

print("All final integrity assertions passed.")
print("Split counts:", sizes)
print("Class coverage:",coverage)
print("Minimum examples per class:",mins)
print("Publisher test membership identical to original:",len(test_orig),"images")
print("Confirmed pair edges:",len(confirmed))
print("Distinct excluded publisher training images:",len(actual_excluded))

## 3. Freeze manifest and provenance

This will write the agreed artifacts into `splits/pakistan/`. The notebook **refuses to overwrite** an existing final artifact whose contents differ, so a future accidental rerun cannot silently change the dataset assignments.

For the main Xception/ConvNeXt comparison, use the *same versioned* final manifest and class map. Record `split_metadata.json`'s SHA-256 manifest fingerprint in every training run's metadata and experiment log.

In [ ]:
final_path = split_dir / "final_manifest.csv"
final_map = split_dir / "class_to_idx.json"
final_meta = split_dir / "split_metadata.json"

def write_unless_different(path, data:bytes):
    if path.exists() and path.read_bytes()!=data:
        raise RuntimeError(f"Refusing to overwrite changed approved artifact: {path}")
    path.write_bytes(data)

final_csv_bytes = candidate_path.read_bytes()
class_json_bytes = json.dumps(class_map,indent=2,ensure_ascii=False,sort_keys=True).encode("utf-8")+b"\n"
info={
  "status":"APPROVED_FOR_MODEL_TRAINING_OVERLAP_SCREENED",
  "source":"Pakistan vehicle dataset 2022 Mendeley Version 1",
  "policy":"preserve publisher test (751 images); exclude confirmed related training images touching test; keep confirmed related training images together",
  "seed":int(cfg["random_seed"]),
  "original_publisher_counts": {"train":3096,"test":751},
  "final_counts":{k:int(v) for k,v in sizes.items()},
  "class_coverage":{k:int(v) for k,v in coverage.items()},
  "minimum_per_class":{k:int(v) for k,v in mins.items()},
  "label_count":48,
  "manifest_sha256":sha256(final_csv_bytes).hexdigest(),
  "class_to_idx_sha256":sha256(class_json_bytes).hexdigest(),
  "review_csv_sha256":sha(review_path),
  "confirmed_review_pairs":int(len(confirmed)),
  "excluded_distinct_training_images":int(len(actual_excluded)),
  "source_candidate_manifest_sha256":sha(original_path),
  "caveat":"Only dHash candidates at Hamming distance <=3 were reviewed; unflagged near-identical video frames may remain.",
}
meta_bytes=json.dumps(info,indent=2,ensure_ascii=False).encode("utf-8")+b"\n"
for p,data in [(final_path,final_csv_bytes),(final_map,class_json_bytes),(final_meta,meta_bytes)]:
    if p.exists() and p.read_bytes()!=data:
        raise RuntimeError(f"Approval artifact already exists but differs: {p}. Resolve manually.")

for p,data in [(final_path,final_csv_bytes),(final_map,class_json_bytes),(final_meta,meta_bytes)]:
    write_unless_different(p,data)
print("FROZEN:",final_path)
print("FROZEN:",final_map)
print("FROZEN:",final_meta)
print("Final manifest SHA-256:",info["manifest_sha256"])
print("Final split sizes:",info["final_counts"])
print("Do not edit these final artifacts; change policies via a separately named split version.")

## 4. Next steps (after successful checks)

1. Commit `splits/pakistan/final_manifest.csv`, `splits/pakistan/class_to_idx.json`, `splits/pakistan/split_metadata.json` and the small, reviewed `splits/pakistan/candidate_pair_review.csv` to Git. **Do not commit raw vehicle photos.**
2. Share the printed counts and final manifest SHA-256 with Project HQ.
3. Load only `train` or `val` in training/validation code, never `excluded_test_overlap`. Use the original **test** only for once-per-model held-out evaluation after validation-based checkpoint selection.
4. To ensure fair Xception vs ConvNeXt experiments, both use this **same frozen manifest**.

**Caveat:** Label integrity and near-duplicate detection may remain imperfect; keep that limitation visible in the report. No model runs have been completed at this point.